# Milestone 1 — Workstream A v1

In [1]:
# imports
import sys
import pandas as pd
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
from src.normalize import normalize_stem

In [2]:
# load and attach normalized stem
data_dir = Path("../data")
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
train["stem"] = train["prompt"].apply(normalize_stem)
test["stem"] = test["prompt"].apply(normalize_stem)
print(train.shape, test.shape)

(2000, 9) (500, 8)


In [3]:
# build train lookup tables and consistency check
opt_cols = ["A", "B", "C", "D", "E"]
train_lookup = {}
stem_to_answer = {}
for _, row in train.iterrows():
    sig = (row["stem"], frozenset({row[c] for c in opt_cols}))
    train_lookup[sig] = row[row["answer"]]
    stem_to_answer[row["stem"]] = row["answer"]
n_answers = train.groupby("stem")["answer"].nunique()
assert (n_answers == 1).all(), "Inconsistent answer labels across train duplicates"
print("lookup signatures:", len(train_lookup))
print("unique stems:", len(stem_to_answer))

lookup signatures: 620
unique stems: 252


In [4]:
# predict for each test row
def predict_label(row):
    sig = (row["stem"], frozenset({row[c] for c in opt_cols}))
    if sig in train_lookup:
        correct_text = train_lookup[sig]
        for c in opt_cols:
            if row[c] == correct_text:
                return c
    return stem_to_answer[row["stem"]]

test["pred"] = test.apply(predict_label, axis=1)
print(test["pred"].value_counts())

pred
C    120
B    114
D     96
A     86
E     84
Name: count, dtype: int64


In [5]:
# train self-recovery sanity check
train["pred"] = train.apply(predict_label, axis=1)
recovery = (train["pred"] == train["answer"]).mean()
print(f"train self-recovery rate: {recovery:.4f}")
assert recovery == 1.0, "Train self-recovery must be 100%"

train self-recovery rate: 1.0000


In [6]:
# build top-3 with label-prior filler
label_order = ["B", "C", "A", "D", "E"]

def top3(pred):
    fillers = [l for l in label_order if l != pred]
    return f"{pred} {fillers[0]} {fillers[1]}"

test["Prediction"] = test["pred"].apply(top3)
test[["id", "pred", "Prediction"]].head()

,id,pred,Prediction
0,1,A,A B C
1,2,B,B C A
2,3,B,B C A
3,4,E,E B C
4,5,C,C B A


In [7]:
# write submission
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test["Prediction"],
})
out_path = data_dir / "submission_m1_v1.csv"
submission.to_csv(out_path, index=False)
print(len(submission), "rows written to", out_path)
submission.head()

500 rows written to ..\data\submission_m1_v1.csv


,ID,Prediction
0,1,A B C
1,2,B C A
2,3,B C A
3,4,E B C
4,5,C B A
